# Bellinge LTC Hydrology Flood Prediction - Reproduction Notebook

This notebook reproduces the publication experiments for the Bellinge flash-flood study. It is intentionally structured as a clean Colab workflow, not as a full exploratory lab log.

Main paths:
- Raw Bellinge input data on Drive: `/content/drive/MyDrive/bellinge/input/downloaded`
- Optional prepared regression archive: `/content/drive/MyDrive/bellinge/bellinge_final_regression.tar.gz`
- Notebook outputs: `/content/drive/MyDrive/bellinge/reproduction_runs/<RUN_ID>/`

Recommended Colab runtime:
- GPU: L4 or T4 for model training
- Disk: at least 85 GB free if `RUN_RAW_DATA_PIPELINE=True`, because the SWMM stage is heavy


## 1. Experiment switches

For a quick verification of the publication tables, keep all heavy flags disabled. For a full reproduction from raw data, enable `RUN_RAW_DATA_PIPELINE`, then enable model training flags.


In [ ]:
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/BuczynskiRafal/ltc-hydrology-flood-prediction.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/ltc-hydrology-flood-prediction")

DRIVE_ROOT = Path("/content/drive/MyDrive/bellinge")
RAW_INPUT_DIR_ON_DRIVE = DRIVE_ROOT / "input" / "downloaded"
PREPARED_REGRESSION_ARCHIVE = DRIVE_ROOT / "bellinge_final_regression.tar.gz"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT_ON_DRIVE = DRIVE_ROOT / "reproduction_runs" / RUN_ID

# Heavy switches
RUN_RAW_DATA_PIPELINE = False
RESTORE_PREPARED_REGRESSION_ARCHIVE = True
RUN_BASELINE_TRAINING = False
RUN_LNN_VARIANT_TRAINING = False
BUILD_PUMP_AWARE_DATASET = False
BUILD_FINAL_TABLES = True
EXPORT_RESULTS_TO_DRIVE = True

BASELINE_MODELS = ["gru", "lstm", "tcn", "mlp"]
LNN_VARIANTS = {
    "best stable LNN": {
        "config": "notebooks/article_materials_20260419/lnn_variants/stable_seed42/config.yaml",
        "artifact_prefix": "lnn_stable_seed42",
    },
    "pump-aware LNN": {
        "config": "notebooks/article_materials_20260419/lnn_variants/pump_aware_global/config.yaml",
        "artifact_prefix": "lnn_pump_aware_global",
    },
    "dual-branch LNN": {
        "config": "notebooks/article_materials_20260419/lnn_variants/dual_branch/config.yaml",
        "artifact_prefix": "lnn_dual_branch",
    },
}

print("RUN_ID:", RUN_ID)
print("RUN_ROOT_ON_DRIVE:", RUN_ROOT_ON_DRIVE)

## 2. Mount Drive, clone repository, and install dependencies

If package installation changes core scientific libraries, Colab may ask for a runtime restart. After restart, run the notebook again from this section.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUN_ROOT_ON_DRIVE.mkdir(parents=True, exist_ok=True)
print("Drive mounted.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

In [ ]:
import subprocess

# Keep NumPy/Pandas/SciPy pinned to the stack used during the reported experiments.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--no-cache-dir",
        "numpy==2.0.2",
        "pandas==2.2.3",
        "scipy==1.15.3",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
    ],
    check=True,
)
print("Python dependencies installed.")

In [ ]:
import importlib

import torch

for module_name in [
    "numpy",
    "pandas",
    "geopandas",
    "swmmio",
    "pyswmm",
    "torch",
    "yaml",
]:
    mod = importlib.import_module(module_name)
    print(f"import ok: {module_name} -> {getattr(mod, '__file__', 'builtin')}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({props.total_memory / 1e9:.1f} GB)")
else:
    print(
        "GPU unavailable. Table reconstruction works, but training will be slow or blocked."
    )

## 3. Shared helper functions


In [ ]:
import shutil
import subprocess
import tarfile
from pathlib import Path

import pandas as pd
import yaml


def run_command(command, *, cwd=REPO_DIR):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)


def latest_file(pattern: str, *, root: Path = REPO_DIR) -> Path:
    matches = sorted(root.glob(pattern), key=lambda path: path.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"No files match pattern: {root / pattern}")
    return matches[-1]


def copy_tree_or_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)


def export_path(src: Path, relative_dst: str):
    if not EXPORT_RESULTS_TO_DRIVE:
        return
    dst = RUN_ROOT_ON_DRIVE / relative_dst
    copy_tree_or_file(src, dst)
    print(f"exported: {src} -> {dst}")


def load_yaml(path: str | Path):
    with open(path, "r") as handle:
        return yaml.safe_load(handle)


def checkpoint_from_config(config_path: str | Path) -> Path:
    config = load_yaml(config_path)
    return REPO_DIR / config["output"]["checkpoint_dir"] / "best_model.pt"


def experiment_root_from_config(config_path: str | Path) -> Path:
    checkpoint_dir = checkpoint_from_config(config_path).parent
    # .../artifacts/experiments/<name>/checkpoints/lnn -> .../artifacts/experiments/<name>
    return checkpoint_dir.parents[1]

## 4. Data restore or raw-data pipeline

Use one of two paths:
- `RESTORE_PREPARED_REGRESSION_ARCHIVE=True` restores `data/final_regression/` from a prepared archive.
- `RUN_RAW_DATA_PIPELINE=True` rebuilds the regression tensors from raw Bellinge files and SWMM simulation outputs.

Pump-aware LNN variants additionally require `data/interim/norm_params.pkl`, which is produced by the raw pipeline at the `normalize_split` stage.


In [ ]:
if RESTORE_PREPARED_REGRESSION_ARCHIVE:
    if PREPARED_REGRESSION_ARCHIVE.exists():
        print("Restoring prepared regression archive:", PREPARED_REGRESSION_ARCHIVE)
        with tarfile.open(PREPARED_REGRESSION_ARCHIVE, "r:gz") as archive:
            archive.extractall(REPO_DIR)
    else:
        print("Prepared regression archive not found:", PREPARED_REGRESSION_ARCHIVE)

required_regression_files = [
    "train_X_reduced.npy",
    "train_y_depths.npy",
    "train_y_overflow.npy",
    "train_flood_mask.npy",
    "val_X_reduced.npy",
    "val_y_depths.npy",
    "val_y_overflow.npy",
    "val_flood_mask.npy",
    "test_X_reduced.npy",
    "test_y_depths.npy",
    "test_y_overflow.npy",
    "test_flood_mask.npy",
    "feature_names_reduced.pkl",
    "target_sensors.pkl",
]
missing = [
    name
    for name in required_regression_files
    if not (REPO_DIR / "data" / "final_regression" / name).exists()
]
print("data/final_regression ready:", not missing)
if missing:
    print("Missing prepared regression files:")
    for name in missing:
        print("-", name)

In [ ]:
RAW_REQUIRED_ENTRIES = [
    "1_Assetdata/Links.shp",
    "1_Assetdata/Manholes.shp",
    "2_cleaned_data",
    "3a_Raingauges/5425_ts.txt",
    "3a_Raingauges/5427_ts.txt",
    "3a_Raingauges/Aabakken_bellinge_vandvaerk_v2_ts.txt",
    "3b_Meterologicalstation",
    "7_SWMM/BellingeSWMM_v021_nopervious.inp",
    "7_SWMM/rg_bellinge_Jun2010_Aug2021.dat",
    "Local_X-band",
    "DMI_C-band",
]


def validate_raw_input(path: Path):
    missing = [rel for rel in RAW_REQUIRED_ENTRIES if not (path / rel).exists()]
    if missing:
        raise FileNotFoundError("Missing raw input entries:\n- " + "\n- ".join(missing))


if RUN_RAW_DATA_PIPELINE:
    validate_raw_input(RAW_INPUT_DIR_ON_DRIVE)
    local_downloaded = REPO_DIR / "downloaded"
    print("Copying raw downloaded data to local Colab disk. This can take a while.")
    shutil.copytree(RAW_INPUT_DIR_ON_DRIVE, local_downloaded, dirs_exist_ok=True)
    validate_raw_input(local_downloaded)
    print("Raw input ready:", local_downloaded)

In [ ]:
RAW_PIPELINE_STAGES = [
    ("load_raw_data", ["python", "-m", "src.load_raw_data"]),
    ("clean_data", ["python", "-m", "src.clean_data"]),
    ("engineer_features", ["python", "-m", "src.engineer_features"]),
    ("normalize_split", ["python", "-m", "src.normalize_split"]),
    ("identify_periods", ["python", "-m", "src.swmm_simulation.identify_periods"]),
    (
        "run_swmm_full_2017_2019",
        [
            "python",
            "-m",
            "src.swmm_simulation.run_swmm_full_2017_2019",
            "--artifacts-dir",
            "data/interim/swmm_runs/historical_2017_2019",
            "--resume",
        ],
    ),
    (
        "define_floods_all_years",
        ["python", "-m", "src.swmm_simulation.define_floods_all_years"],
    ),
    ("create_labels", ["python", "-m", "src.create_labels"]),
    ("prepare_windows", ["python", "-m", "src.prepare_windows"]),
    ("select_features", ["python", "-m", "src.select_features"]),
]

if RUN_RAW_DATA_PIPELINE:
    for stage_name, command in RAW_PIPELINE_STAGES:
        print("\n" + "=" * 90)
        print("DATA STAGE:", stage_name)
        run_command(command)
        export_path(
            REPO_DIR / "data" / "interim",
            f"pipeline_snapshots/{stage_name}/data_interim",
        )
        if (REPO_DIR / "data" / "final_regression").exists():
            export_path(
                REPO_DIR / "data" / "final_regression",
                f"pipeline_snapshots/{stage_name}/data_final_regression",
            )
    print("Raw-data pipeline complete.")

## 5. Build pump-aware dataset

Required for `pump-aware LNN` and `dual-branch LNN`. This step augments the 31-channel reduced tensor with 5 pump logic features derived from the G80F13P pump thresholds.


In [ ]:
pump_aware_dir = REPO_DIR / "data" / "final_regression_pump_aware"
pump_aware_ready = (pump_aware_dir / "train_X_reduced.npy").exists()

if BUILD_PUMP_AWARE_DATASET or (RUN_LNN_VARIANT_TRAINING and not pump_aware_ready):
    norm_params = REPO_DIR / "data" / "interim" / "norm_params.pkl"
    if not norm_params.exists():
        raise FileNotFoundError(
            "data/interim/norm_params.pkl is required to build pump-aware tensors. "
            "Run the raw-data pipeline through normalize_split, restore data/interim from Drive, "
            "or restore data/final_regression_pump_aware directly before training pump-aware variants."
        )
    run_command(["python", "-m", "experiments.build_pump_aware_dataset"])
    export_path(pump_aware_dir, "data/final_regression_pump_aware")
elif pump_aware_ready:
    print("Pump-aware dataset already exists:", pump_aware_dir)
else:
    print("Skipping pump-aware dataset build.")

## 6. Train and evaluate baseline models

This trains GRU, LSTM, TCN, and MLP using canonical configs, selects the overflow threshold on validation data, and evaluates on the test split.


In [ ]:
if RUN_BASELINE_TRAINING:
    for model_name in BASELINE_MODELS:
        print("\n" + "=" * 90)
        print("BASELINE:", model_name.upper())
        run_command(["python", "-m", "experiments.cli", "train", model_name])
        threshold_path = (
            REPO_DIR / "artifacts" / "thresholds" / f"{model_name}_val_threshold.json"
        )
        run_command(
            [
                "python",
                "-m",
                "experiments.select_threshold_on_val",
                "--model",
                model_name,
                "--output",
                str(threshold_path),
            ]
        )
        run_command(
            [
                "python",
                "-m",
                "experiments.evaluate_release",
                "--model",
                model_name,
                "--threshold-artifact",
                str(threshold_path),
                "--results-dir",
                "artifacts/results/release",
            ]
        )
        export_path(
            REPO_DIR / "artifacts" / "checkpoints" / model_name,
            f"baselines/{model_name}/checkpoint",
        )
    export_path(
        REPO_DIR / "artifacts" / "results" / "release", "baselines/release_results"
    )
else:
    print("Skipping baseline training.")

## 7. Train and evaluate LNN variants

The publication variants are:
- `best stable LNN`: depth-only, separate depth heads, per-neuron tau.
- `pump-aware LNN`: same model with pump logic features in the shared encoder.
- `dual-branch LNN`: final model with a dedicated pump encoder branch.


In [ ]:
if RUN_LNN_VARIANT_TRAINING:
    for label, spec in LNN_VARIANTS.items():
        config_path = REPO_DIR / spec["config"]
        checkpoint_path = checkpoint_from_config(config_path)
        exp_root = experiment_root_from_config(config_path)
        results_dir = exp_root / "results" / "release"
        predictions_dir = exp_root / "results" / "predictions"

        print("\n" + "=" * 90)
        print("LNN VARIANT:", label)
        print("config:", config_path)
        run_command(
            [
                "python",
                "-m",
                "experiments.cli",
                "train",
                "lnn",
                "--config",
                str(config_path),
            ]
        )
        run_command(
            [
                "python",
                "-m",
                "experiments.evaluate_depth_only_release",
                "--model",
                "lnn",
                "--config",
                str(config_path),
                "--checkpoint",
                str(checkpoint_path),
                "--results-dir",
                str(results_dir),
                "--predictions-dir",
                str(predictions_dir),
                "--artifact-prefix",
                spec["artifact_prefix"],
            ]
        )
        export_path(exp_root, f"lnn_variants/{label.replace(' ', '_')}")
else:
    print("Skipping LNN variant training.")

## 8. Build final comparison tables

If models were trained in this notebook, the cell uses fresh metrics from `artifacts/`. If not, it uses the manuscript snapshot already included in `notebooks/article_materials_20260419/`.


In [ ]:
def included_metric_path(model: str) -> Path:
    paths = {
        "GRU": "notebooks/article_materials_20260419/baselines/GRU/gru_test_20260416_121123_metrics.json",
        "LSTM": "notebooks/article_materials_20260419/baselines/LSTM/lstm_test_20260416_121317_metrics.json",
        "TCN": "notebooks/article_materials_20260419/baselines/TCN/tcn_test_20260416_121518_metrics.json",
        "MLP": "notebooks/article_materials_20260419/baselines/MLP/mlp_test_20260416_121641_metrics.json",
    }
    return REPO_DIR / paths[model]


def fresh_baseline_metric_path(model_name: str) -> Path:
    return latest_file(f"artifacts/results/release/{model_name}_test_*_metrics.json")


def fresh_lnn_metric_path(config_path: str, prefix: str) -> Path:
    exp_root = experiment_root_from_config(REPO_DIR / config_path)
    return latest_file(
        f"{exp_root.relative_to(REPO_DIR)}/results/release/{prefix}_test_*_metrics.json"
    )


def resolve_model_metrics():
    metric_specs = []
    for model_name in BASELINE_MODELS:
        label = model_name.upper()
        path = (
            fresh_baseline_metric_path(model_name)
            if RUN_BASELINE_TRAINING
            else included_metric_path(label)
        )
        metric_specs.append((label, path))

    if RUN_LNN_VARIANT_TRAINING:
        for label, spec in LNN_VARIANTS.items():
            metric_specs.append(
                (label, fresh_lnn_metric_path(spec["config"], spec["artifact_prefix"]))
            )
    else:
        # The included article benchmark already contains the final LNN metrics.
        # If you need fresh LNN metrics without retraining, point these paths to Drive exports.
        benchmark_dir = (
            REPO_DIR / "notebooks" / "article_materials_20260419" / "benchmark"
        )
        print("Using included article benchmark tables from:", benchmark_dir)
        return None
    return metric_specs


if BUILD_FINAL_TABLES:
    metric_specs = resolve_model_metrics()
    if metric_specs is None:
        article_table = pd.read_csv(
            REPO_DIR
            / "notebooks/article_materials_20260419/benchmark/article_main_table.csv"
        )
        grouped = pd.read_csv(
            REPO_DIR
            / "notebooks/article_materials_20260419/benchmark/grouped_summary.csv"
        )
        display(article_table)
        display(grouped)
        export_path(
            REPO_DIR / "notebooks/article_materials_20260419/benchmark",
            "final_tables/included_benchmark",
        )
    else:
        output_dir = (
            REPO_DIR / "notebooks" / "release" / f"final_article_benchmark_{RUN_ID}"
        )
        command = ["python", "-m", "experiments.final_model_comparison"]
        for label, path in metric_specs:
            print(label, "->", path)
            command.extend(["--model", f"{label}={path}"])
        command.extend(["--output-dir", str(output_dir)])
        if EXPORT_RESULTS_TO_DRIVE:
            command.extend(
                ["--copy-to", str(RUN_ROOT_ON_DRIVE / "final_article_benchmark")]
            )
        run_command(command)
        display(pd.read_csv(output_dir / "article_main_table.csv"))
        display(pd.read_csv(output_dir / "grouped_summary.csv"))
else:
    print("Skipping final table generation.")

## 9. Smoke tests and artifact summary


In [ ]:
run_command(["python", "-m", "pytest", "tests/test_env.py", "tests/test_cli.py", "-q"])

In [ ]:
print("Repository:", REPO_DIR)
print("Drive run folder:", RUN_ROOT_ON_DRIVE)
print(
    "Included manuscript materials:", REPO_DIR / "notebooks/article_materials_20260419"
)

if RUN_ROOT_ON_DRIVE.exists():
    print("\nDrive artifacts:")
    for path in sorted(RUN_ROOT_ON_DRIVE.rglob("*")):
        if path.is_file():
            print(path.relative_to(RUN_ROOT_ON_DRIVE))